# Phase 7: SchNet Optuna 300k (Colab A100)

1. Upload `pyg_3d_graphs_etkdg_300k.pt` to Google Drive (`MyDrive/molgap/`)
2. Runtime -> Change runtime type -> A100
3. Run All

In [1]:
!pip install -q torch-geometric optuna
!pip install -q pyg-lib -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])")+cu$(python -c "import torch; print(torch.version.cuda.replace('.',''))").html

: 

In [2]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
import os, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingWarmRestarts
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

torch.backends.cudnn.benchmark = True

props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name}, {props.total_memory / 1e9:.1f} GB")
print(f"torch {torch.__version__}, CUDA {torch.version.cuda}")

In [4]:
# ====== CONFIG ======
DRIVE_DIR = "/content/drive/MyDrive/MolGap"
GRAPH_PATH = f"{DRIVE_DIR}/pyg_3d_graphs_etkdg_300k.pt"
SAVE_DIR = f"{DRIVE_DIR}/phase7_results"
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 42
TARGET_COLS = ["homo", "lumo", "gap"]
N_TRIALS = 20
FAST_EPOCHS = 80
FULL_EPOCHS = 500
FULL_PATIENCE = 40

# A100: larger batch size
NUM_WORKERS = 4
PIN_MEMORY = True

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda")

In [5]:
# ====== SchNet ======
class SchNetWrapper(nn.Module):
    def __init__(self, hidden_channels, num_filters, num_interactions,
                 num_gaussians, cutoff, dropout=0.1, n_targets=3,
                 use_charges=False):
        super().__init__()
        from torch_geometric.nn.models import SchNet
        self.schnet = SchNet(
            hidden_channels=hidden_channels, num_filters=num_filters,
            num_interactions=num_interactions, num_gaussians=num_gaussians,
            cutoff=cutoff,
        )
        self.use_charges = use_charges
        if use_charges:
            self.charge_proj = nn.Linear(1, hidden_channels)
        self.head = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.SiLU(),
            nn.Linear(hidden_channels // 2, n_targets),
        )

    def forward(self, z, pos, batch, charges=None):
        from torch_geometric.nn import global_mean_pool
        from torch_geometric.nn.models.schnet import radius_graph
        h = self.schnet.embedding(z)
        if self.use_charges and charges is not None:
            h = h + self.charge_proj(charges.unsqueeze(-1))
        edge_index = radius_graph(pos, r=self.schnet.cutoff, batch=batch, max_num_neighbors=32)
        row, col = edge_index
        edge_weight = (pos[row] - pos[col]).norm(dim=-1)
        edge_attr = self.schnet.distance_expansion(edge_weight)
        for interaction in self.schnet.interactions:
            h = h + interaction(h, edge_index, edge_weight, edge_attr)
        h = global_mean_pool(h, batch)
        return self.head(h)

In [6]:
# ====== Load data ======
print("Loading graphs...")
data_list = torch.load(GRAPH_PATH, weights_only=False)
print(f"Loaded {len(data_list)} graphs")

has_charges = hasattr(data_list[0], 'charges')
print(f"Gasteiger charges: {has_charges}")

all_idx = np.arange(len(data_list))
train_valid_idx, test_idx = train_test_split(all_idx, test_size=0.1, random_state=SEED)
train_idx, valid_idx = train_test_split(train_valid_idx, test_size=0.1/0.9, random_state=SEED)

train_data = [data_list[i] for i in train_idx]
valid_data = [data_list[i] for i in valid_idx]
test_data  = [data_list[i] for i in test_idx]

train_y = np.stack([d.y.squeeze(0).numpy() for d in train_data])
y_mean = train_y.mean(axis=0)
y_std = train_y.std(axis=0)
y_std[y_std < 1e-6] = 1.0

for d in train_data + valid_data + test_data:
    d.y = (d.y - torch.tensor(y_mean)) / torch.tensor(y_std)

del data_list
n_total = len(train_data) + len(valid_data) + len(test_data)
print(f"Split: train={len(train_data)}, valid={len(valid_data)}, test={len(test_data)}")

In [7]:
# ====== Training functions ======
def train_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss, n = 0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            charges = getattr(batch, 'charges', None)
            out = model(batch.z, batch.pos, batch.batch, charges=charges)
            loss = F.l1_loss(out, batch.y)
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * batch.num_graphs
        n += batch.num_graphs
    return total_loss / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        batch = batch.to(device)
        with torch.amp.autocast("cuda"):
            charges = getattr(batch, 'charges', None)
            out = model(batch.z, batch.pos, batch.batch, charges=charges)
        preds.append(out.cpu().numpy())
        trues.append(batch.y.cpu().numpy())
    return np.vstack(preds), np.vstack(trues)


def run_training(params, train_loader, valid_loader,
                 max_epochs, patience, verbose=False,
                 save_ckpt=False, resume_ckpt=None):
    model = SchNetWrapper(
        hidden_channels=params["hidden_channels"],
        num_filters=params["num_filters"],
        num_interactions=params["num_interactions"],
        num_gaussians=params["num_gaussians"],
        cutoff=params["cutoff"],
        dropout=params["dropout"],
        use_charges=has_charges,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=params["lr"], weight_decay=params["weight_decay"])
    if params["scheduler"] == "cosine":
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)
    else:
        scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5,
                                      patience=max(5, patience // 3), min_lr=1e-6)
    scaler = torch.amp.GradScaler("cuda")

    best_val_mae = float("inf")
    best_epoch = 0
    best_state = None
    log_rows = []
    start_epoch = 1

    # Resume from checkpoint
    if resume_ckpt and os.path.exists(resume_ckpt):
        ckpt = torch.load(resume_ckpt, weights_only=False)
        model.load_state_dict({k: v.to(device) for k, v in ckpt["model_state_dict"].items()})
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        best_val_mae = ckpt["best_val_mae"]
        best_epoch = ckpt["best_epoch"]
        start_epoch = ckpt["epoch"] + 1
        log_rows = ckpt.get("log_rows", [])
        if verbose:
            print(f"  Resumed from epoch {start_epoch}, best MAE={best_val_mae:.4f}@{best_epoch}")

    for epoch in range(start_epoch, max_epochs + 1):
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, scaler)

        if not np.isfinite(train_loss):
            if verbose:
                print(f"  NaN at epoch {epoch}, stopping.")
            break

        val_pred, val_true = evaluate(model, valid_loader)
        val_pred_real = val_pred * y_std + y_mean
        val_true_real = val_true * y_std + y_mean
        val_mae = float(np.mean(np.abs(val_pred_real - val_true_real)))

        if params["scheduler"] == "cosine":
            scheduler.step(epoch)
        else:
            scheduler.step(val_mae)

        elapsed = time.time() - t0
        log_rows.append({"epoch": epoch, "train_loss": train_loss,
                         "val_mae": val_mae, "lr": optimizer.param_groups[0]["lr"],
                         "time_s": elapsed})

        is_best = val_mae < best_val_mae
        if is_best:
            best_val_mae = val_mae
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if verbose and (epoch % 10 == 0 or epoch == 1 or is_best):
            marker = " *" if is_best else ""
            print(f"  Epoch {epoch:3d} | loss={train_loss:.4f} | val_MAE={val_mae:.4f} | "
                  f"best={best_val_mae:.4f}@{best_epoch} | "
                  f"lr={optimizer.param_groups[0]['lr']:.1e} | {elapsed:.1f}s{marker}")

        if save_ckpt and epoch % 5 == 0:
            ckpt = {
                "epoch": epoch,
                "model_state_dict": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "best_val_mae": best_val_mae,
                "best_epoch": best_epoch,
                "params": params,
                "y_mean": y_mean.tolist(),
                "y_std": y_std.tolist(),
                "log_rows": log_rows,
            }
            torch.save(ckpt, f"{SAVE_DIR}/checkpoint.pt")
            if is_best:
                torch.save(best_state, f"{SAVE_DIR}/best_model.pt")

        if epoch - best_epoch >= patience:
            if verbose:
                print(f"  Early stop at epoch {epoch} (best={best_epoch})")
            break

    if save_ckpt and best_state is not None:
        torch.save(best_state, f"{SAVE_DIR}/best_model.pt")
        pd.DataFrame(log_rows).to_csv(f"{SAVE_DIR}/retrain_log_300k.csv", index=False)

    return best_val_mae, best_epoch, best_state, log_rows

In [8]:
# ====== Optuna (20 trials, 80 epochs each) ======
import optuna

# Resume Optuna if previous study exists
OPTUNA_DB = f"sqlite:///{SAVE_DIR}/optuna_300k.db"

def objective(trial):
    params = {
        "hidden_channels": trial.suggest_categorical("hidden_channels", [128, 192, 256]),
        "num_filters": trial.suggest_categorical("num_filters", [128, 192, 256]),
        "num_interactions": trial.suggest_int("num_interactions", 3, 6),
        "num_gaussians": trial.suggest_categorical("num_gaussians", [25, 50, 100]),
        "cutoff": trial.suggest_categorical("cutoff", [5.0, 6.0, 8.0, 10.0]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3, step=0.05),
        "lr": trial.suggest_float("lr", 1e-4, 1e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128, 256]),
        "scheduler": trial.suggest_categorical("scheduler", ["plateau", "cosine"]),
    }

    loader_t = DataLoader(train_data, batch_size=params["batch_size"], shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    loader_v = DataLoader(valid_data, batch_size=params["batch_size"],
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    try:
        best_mae, best_ep, _, _ = run_training(
            params, loader_t, loader_v,
            max_epochs=FAST_EPOCHS, patience=15)
    except Exception as e:
        print(f"  Trial {trial.number} failed: {e}")
        torch.cuda.empty_cache()
        return float("inf")

    print(f"  Trial {trial.number:2d} | MAE={best_mae:.4f} @ ep{best_ep} | "
          f"h={params['hidden_channels']} int={params['num_interactions']} "
          f"cut={params['cutoff']:.0f} lr={params['lr']:.1e} bs={params['batch_size']}")
    torch.cuda.empty_cache()
    return best_mae


study = optuna.create_study(
    study_name="schnet_300k",
    direction="minimize",
    storage=OPTUNA_DB,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED),
)

done = len(study.trials)
remaining = max(0, N_TRIALS - done)
print(f"Optuna: {done} trials done, {remaining} remaining")

if remaining > 0:
    study.optimize(objective, n_trials=remaining)

print(f"\nBest trial {study.best_trial.number}: MAE={study.best_value:.4f}")
print(f"Params: {study.best_params}")

study.trials_dataframe().to_csv(f"{SAVE_DIR}/optuna_trials_300k.csv", index=False)
with open(f"{SAVE_DIR}/optuna_best_params_300k.json", "w") as f:
    json.dump(study.best_params, f, indent=2)

In [ ]:
# ====== Full retrain with best params ======
print(f"Full retrain: {FULL_EPOCHS} epochs, patience={FULL_PATIENCE}")

best_params = dict(study.best_params)
loader_t = DataLoader(train_data, batch_size=best_params["batch_size"], shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
loader_v = DataLoader(valid_data, batch_size=best_params["batch_size"],
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# Resume if checkpoint exists
ckpt_path = f"{SAVE_DIR}/checkpoint.pt"
resume = ckpt_path if os.path.exists(ckpt_path) else None

best_mae, best_epoch, best_state, log_rows = run_training(
    best_params, loader_t, loader_v,
    max_epochs=FULL_EPOCHS, patience=FULL_PATIENCE,
    verbose=True, save_ckpt=True, resume_ckpt=resume)

print(f"\nBest val MAE: {best_mae:.4f} @ epoch {best_epoch}")

In [ ]:
# ====== Test evaluation ======
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = SchNetWrapper(
    hidden_channels=best_params["hidden_channels"],
    num_filters=best_params["num_filters"],
    num_interactions=best_params["num_interactions"],
    num_gaussians=best_params["num_gaussians"],
    cutoff=best_params["cutoff"],
    dropout=best_params["dropout"],
    use_charges=has_charges,
).to(device)
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

test_loader = DataLoader(test_data, batch_size=best_params["batch_size"],
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_pred, test_true = evaluate(model, test_loader)
test_pred_real = test_pred * y_std + y_mean
test_true_real = test_true * y_std + y_mean

print(f"\n{'='*60}")
print(f"  Phase 7: SchNet {n_total//1000}k Test Results")
print(f"{'='*60}")

metrics = {}
for i, t in enumerate(TARGET_COLS):
    mae = mean_absolute_error(test_true_real[:, i], test_pred_real[:, i])
    rmse = np.sqrt(mean_squared_error(test_true_real[:, i], test_pred_real[:, i]))
    r2 = r2_score(test_true_real[:, i], test_pred_real[:, i])
    metrics[t] = {"mae": float(mae), "rmse": float(rmse), "r2": float(r2)}
    print(f"  {t:5s}: MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")

avg_mae = np.mean([m["mae"] for m in metrics.values()])
avg_r2 = np.mean([m["r2"] for m in metrics.values()])
print(f"  avg  : MAE={avg_mae:.4f}  R2={avg_r2:.4f}")
print(f"\n  vs Phase 6 (44.8k): avg MAE=0.162, R2=0.882")

In [ ]:
# ====== Save final outputs ======
torch.save(best_state, f"{SAVE_DIR}/gnn_schnet_3d_optuna_300k.pt")

final_meta = {
    "model": "SchNet_3D_ETKDG_300k",
    "params": best_params,
    "n_data": n_total,
    "mw_range": "200-1000",
    "best_epoch": best_epoch,
    "epochs_trained": len(log_rows),
    "n_trials": N_TRIALS,
    "metrics": metrics,
    "y_mean": y_mean.tolist(),
    "y_std": y_std.tolist(),
}
with open(f"{SAVE_DIR}/schnet_optuna_300k_metrics.json", "w") as f:
    json.dump(final_meta, f, indent=2)

print(f"All saved to {SAVE_DIR}/")
print(f"Download: gnn_schnet_3d_optuna_300k.pt -> models/")